# TIGER ROI classifier — training

Trains the tissue-compartment (ROI) classifier on the TIGER dataset.
- Assumes BCSS raw PNG → 512px tile extraction has already been done, producing
  `train_df_tiger_updated_512.csv` / `val_df_tiger_updated_512.csv` / `test_df_tiger_updated_512.csv`
  (see `TIGER_training/prepare_tiger_tiles.py`), each with at least a `data` column (path to the
  tile image) and a `label` column (see label mapping below).
- Uses a feature-extractor checkpoint (`FEATURE_EXTRACTOR_DIR` below) — a DINO-pretrained
  ConvNeXt-Base backbone in this codebase, but any backbone matching `model.py`'s
  `StudentModel_convnext` state-dict layout works; substitute your own if you don't have this one.
- **Checkpoint selection uses the validation split only.** The test split is held out and
  evaluated exactly once, after training, using the checkpoint the validation loss selected — so
  the reported test metrics are not the same set of examples used to pick the model.

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import LabelBinarizer
from tensorboardX import SummaryWriter

import sys
project_root = os.path.dirname(os.path.abspath('.'))
sys.path.append(project_root)

from model import Student_convnext_backbone, Student_Projection_Head, StudentModel_convnext, Patch_Classifier_Softmax_KD

In [2]:
# ---- Config ----
FEATURE_EXTRACTOR_DIR = '../G2B_BRCA.pth'  # path to your feature-extractor checkpoint (see markdown above)
TRAIN_CSV = './train_df_tiger_updated_512.csv'
VAL_CSV = './val_df_tiger_updated_512.csv'
TEST_CSV = './test_df_tiger_updated_512.csv'

# Tile-extraction pipelines typically write the CSV's 'data' column as an absolute path on
# whatever machine extracted the tiles. If you're running this on a different machine (e.g. you
# copied the tile directory elsewhere), set OLD_DATA_ROOT to the path prefix actually stored in
# the CSV and NEW_DATA_ROOT to where the tiles live on this machine. Leave both as '' to skip
# remapping (i.e. the CSV's paths are already correct as-is).
OLD_DATA_ROOT = ''
NEW_DATA_ROOT = ''

DEVICE = 'cuda:0'  # set to any free GPU on your machine
BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 25
NUM_CLASSES = 5
LR = 1e-6
WEIGHT_DECAY = 1e-3

WEIGHTS_DIR = './tiger_weights'
TB_DIR = './tensorboard_log_tiger'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(TB_DIR, exist_ok=True)

device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda:0


In [3]:
# ---- Feature extractor (DINO-pretrained ConvNeXt-Base backbone) ----
student = Student_convnext_backbone()
student_projection_head = Student_Projection_Head(
    in_dim=1024, out_dim=1536, use_bn=True, use_mid_layer=False,
    hidden_dim=2048, bottleneck_dim=256, nlayers=3, logger=None,
)
student_final = StudentModel_convnext(backbone=student, projection_head=student_projection_head)
student_final.load_state_dict(torch.load(FEATURE_EXTRACTOR_DIR, map_location='cpu')['student'])

backbone = student_final.backbone
model = Patch_Classifier_Softmax_KD(backbone=backbone, num_classes=NUM_CLASSES).to(device)

## Label mapping

Invasive Tumor + In-situ Tumor → 0, Tumor-associated Stroma → 1, Necrosis → 2, Inflamed Stroma → 3, Rest → 4
(already reflected in the CSV's `label` column — shown here for reference)

In [4]:
label_dict = {'1': 0, '2': 1, '3': 0, '5': 2, '6': 3, '7': 4}
label_dict

{'1': 0, '2': 1, '3': 0, '5': 2, '6': 3, '7': 4}

In [5]:
data_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5))], p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517)),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517)),
])

In [6]:
class TigerTileDataset(Dataset):
    def __init__(self, csv_path, transform):
        df = pd.read_csv(csv_path, index_col=False)
        if OLD_DATA_ROOT:
            df['data'] = df['data'].str.replace(OLD_DATA_ROOT, NEW_DATA_ROOT, regex=False)
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_path = self.df.loc[idx, 'data']
        label = self.df.loc[idx, 'label']
        patch = Image.open(file_path).convert('RGB')
        patch = self.transform(patch)
        label = torch.tensor(label, dtype=torch.long)
        return patch, label


train_dataset = TigerTileDataset(TRAIN_CSV, transform=data_transform)
val_dataset = TigerTileDataset(VAL_CSV, transform=test_transform)
test_dataset = TigerTileDataset(TEST_CSV, transform=test_transform)

# sanity-check that the data actually exists
assert os.path.isfile(train_dataset.df.loc[0, 'data']), train_dataset.df.loc[0, 'data']
print('train:', len(train_dataset), 'val:', len(val_dataset), 'test:', len(test_dataset))
print(train_dataset.df['label'].value_counts().sort_index())
print(val_dataset.df['label'].value_counts().sort_index())
print(test_dataset.df['label'].value_counts().sort_index())

train: 6052 val: 1297 test: 1297
label
0    2813
1    2215
2     325
3     598
4     101
Name: count, dtype: int64
label
0    602
1    475
2     70
3    128
4     22
Name: count, dtype: int64
label
0    603
1    475
2     69
3    128
4     22
Name: count, dtype: int64


In [7]:
# WeightedRandomSampler to correct for class imbalance (uses the CSV's label column directly —
# no separate targets.npy cache needed)
train_targets = train_dataset.df['label'].to_numpy()
class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_targets]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                           pin_memory=True, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                         pin_memory=True, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=True, shuffle=False)

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
writer = SummaryWriter(TB_DIR, flush_secs=15)

## Training loop

At each epoch, compute train/validation metrics (loss, accuracy, macro/weighted F1, ROC-AUC), and
**save `best_model.pth` only when validation loss improves on the previous best** (the original
notebook had this best-model selection logic commented out, so it just accumulated a checkpoint
per epoch). The last epoch is also saved separately as `last_model.pth`. The test split is not
touched here at all — see the final cell below for the one-time test evaluation.

In [9]:
def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_probs, all_preds, all_targets = [], [], []

    for patch, label in tqdm(loader, leave=False):
        patch = patch.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)

        if train:
            logits, probs, preds = model(patch)
            loss = criterion(logits, label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                logits, probs, preds = model(patch)
                loss = criterion(logits, label)

        total_loss += loss.item()
        all_probs.append(probs.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(label.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

    try:
        lb = LabelBinarizer()
        targets_bin = lb.fit_transform(all_targets)
        if len(np.unique(all_targets)) > 1:
            if len(np.unique(all_targets)) == 2:
                auc = roc_auc_score(all_targets, all_probs[:, 1])
            else:
                auc = roc_auc_score(targets_bin, all_probs, multi_class='ovr', average='macro')
        else:
            auc = 0.0
    except ValueError as e:
        print('AUC calculation error:', e)
        auc = 0.0

    return {'loss': avg_loss, 'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1, 'auc': auc}

In [10]:
best_val_loss = float('inf')
best_epoch = -1
history = []

for epoch in range(EPOCHS):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)

    print(f"Epoch {epoch} - Train: loss {train_metrics['loss']:.4f} acc {train_metrics['accuracy']:.4f} "
          f"macroF1 {train_metrics['macro_f1']:.4f} AUC {train_metrics['auc']:.4f}")
    print(f"Epoch {epoch} - Val:   loss {val_metrics['loss']:.4f} acc {val_metrics['accuracy']:.4f} "
          f"macroF1 {val_metrics['macro_f1']:.4f} AUC {val_metrics['auc']:.4f}")

    for split, m in (('Train', train_metrics), ('Val', val_metrics)):
        writer.add_scalar(f'{split}/Loss', m['loss'], epoch)
        writer.add_scalar(f'{split}/Accuracy', m['accuracy'], epoch)
        writer.add_scalar(f'{split}/Macro_F1', m['macro_f1'], epoch)
        writer.add_scalar(f'{split}/Weighted_F1', m['weighted_f1'], epoch)
        writer.add_scalar(f'{split}/ROC-AUC', m['auc'], epoch)

    history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()},
                     **{f'val_{k}': v for k, v in val_metrics.items()}})

    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_epoch = epoch
        torch.save({'epoch': epoch, 'state_dict': model.state_dict(), 'val_metrics': val_metrics},
                    os.path.join(WEIGHTS_DIR, 'best_model.pth'))
        print(f'  -> new best model saved (epoch {epoch}, val loss {best_val_loss:.4f})')

    print('-' * 80)

torch.save({'epoch': EPOCHS - 1, 'state_dict': model.state_dict(), 'val_metrics': val_metrics},
            os.path.join(WEIGHTS_DIR, 'last_model.pth'))

pd.DataFrame(history).to_csv(os.path.join(WEIGHTS_DIR, 'training_history.csv'), index=False)
print(f'Best epoch: {best_epoch}, best val loss: {best_val_loss:.4f}')
print(f'Best model saved to {os.path.join(WEIGHTS_DIR, "best_model.pth")}')

Epoch 0 - Train: loss 1.4971 acc 0.5195 macroF1 0.5061 AUC 0.8415
Epoch 0 - Val:   loss 1.3310 acc 0.7209 macroF1 0.6379 AUC 0.9369
  -> new best model saved (epoch 0, val loss 1.3310)
--------------------------------------------------------------------------------
Epoch 1 - Train: loss 1.1397 acc 0.7748 macroF1 0.7684 AUC 0.9418
Epoch 1 - Val:   loss 0.8579 acc 0.7579 macroF1 0.6752 AUC 0.9581
  -> new best model saved (epoch 1, val loss 0.8579)
--------------------------------------------------------------------------------
Epoch 2 - Train: loss 0.6319 acc 0.8275 macroF1 0.8278 AUC 0.9632
Epoch 2 - Val:   loss 0.5260 acc 0.8150 macroF1 0.7435 AUC 0.9650
  -> new best model saved (epoch 2, val loss 0.5260)
--------------------------------------------------------------------------------
Epoch 3 - Train: loss 0.3911 acc 0.8710 macroF1 0.8704 AUC 0.9791
Epoch 3 - Val:   loss 0.4939 acc 0.8350 macroF1 0.7689 AUC 0.9683
  -> new best model saved (epoch 3, val loss 0.4939)
-----------------

## Final test evaluation (one-time)

Loads the checkpoint selected by validation loss above and evaluates it on the held-out test
split exactly once. These are the numbers that should be reported as the classifier's
performance -- unlike the per-epoch validation metrics above, this test split played no role
in choosing which checkpoint to use.

In [11]:
best_ckpt = torch.load(os.path.join(WEIGHTS_DIR, 'best_model.pth'), map_location=device)
model.load_state_dict(best_ckpt['state_dict'])
print(f"Loaded best_model.pth (selected at epoch {best_ckpt['epoch']}, "
      f"val loss {best_ckpt['val_metrics']['loss']:.4f})")

model.eval()
all_probs, all_preds, all_targets = [], [], []
total_loss = 0.0
with torch.no_grad():
    for patch, label in tqdm(test_loader, leave=False):
        patch = patch.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        logits, probs, preds = model(patch)
        loss = criterion(logits, label)
        total_loss += loss.item()
        all_probs.append(probs.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(label.detach().cpu().numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)
avg_loss = total_loss / len(test_loader)

acc = accuracy_score(all_targets, all_preds)
macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
weighted_f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0)
targets_bin = LabelBinarizer().fit_transform(all_targets)
auc = roc_auc_score(targets_bin, all_probs, multi_class='ovr', average='macro')

print(f"Test: loss {avg_loss:.4f} acc {acc:.4f} macroF1 {macro_f1:.4f} weightedF1 {weighted_f1:.4f} AUC {auc:.4f}")


def bootstrap_macro_auc_ci(targets, probs, n_bootstrap=2000, ci=95, random_state=42):
    rng = np.random.RandomState(random_state)
    n = len(targets)
    boots = []
    for _ in range(n_bootstrap):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        yt, yp = targets[idx], probs[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            yt_bin = LabelBinarizer().fit_transform(yt)
            if yt_bin.shape[1] < 2:
                continue
            boots.append(roc_auc_score(yt_bin, yp, multi_class='ovr', average='macro'))
        except ValueError:
            continue
    boots = np.array(boots)
    alpha = (100 - ci) / 2
    lo, hi = np.percentile(boots, [alpha, 100 - alpha])
    return lo, hi, len(boots)


auc_lo, auc_hi, n_valid = bootstrap_macro_auc_ci(all_targets, all_probs, n_bootstrap=2000, random_state=42)
print(f"Test AUC 95% CI (bootstrap, n_valid={n_valid}): [{auc_lo:.4f}, {auc_hi:.4f}]")

test_metrics = {'loss': avg_loss, 'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1, 'auc': auc}
for k, v in test_metrics.items():
    writer.add_scalar(f'Test/{k}', v, best_ckpt['epoch'])

Loaded best_model.pth (selected at epoch 24, val loss 0.3096)
Test: loss 0.2370 acc 0.9136 macroF1 0.9009 weightedF1 0.9141 AUC 0.9911
Test AUC 95% CI (bootstrap, n_valid=2000): [0.9885, 0.9936]
